# TripAI NLP Intent Extraction: QLoRA Fine-Tuning Notebook

This notebook contains the complete training code to fine-tune Qwen2.5-3B-Instruct on the TripAI conversational travel intent dataset using **Unsloth** and **QLoRA** on a free Google Colab T4 GPU.

### 1. Install Dependencies
We install Unsloth (which accelerates training on T4 GPU) and other Hugging Face libraries like `trl`, `peft`, `transformers`, etc.

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.26" trl peft transformers accelerate

### 2. Load Base Model
We load the 4-bit quantized base model `Qwen2.5-3B-Instruct`.

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None  # Auto detection
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-3B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

### 3. Setup LoRA Adapters
We configure PEFT/LoRA targeting key attention projection layers.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

### 4. Load & Format Dataset
Upload the generated `training_data.json` dataset to your Colab workspace and load it below.

In [ ]:
from datasets import load_dataset

# Formats the instruction, query, and expected output in Qwen Chat Format
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input_text, output in zip(instructions, inputs, outputs):
        text = f"<|im_start|>system\n{instruction}<|im_end|>\n<|im_start|>user\n{input_text}<|im_end|>\n<|im_start|>assistant\n{output}<|im_end|>"
        texts.append(text)
    return { "text" : texts }

# Make sure to upload 'training_data.json' to Colab folder first!
dataset = load_dataset("json", data_files="training_data.json", split="train")
dataset = dataset.map(formatting_prompts_func, batched=True)

### 5. Setup Trainer and Start Fine-tuning
We use TRL's `SFTTrainer` for instruction tuning.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # Increase to 120-200 for better convergence
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

### 6. Validation (Inference Test)

In [ ]:
FastLanguageModel.for_inference(model)

system_prompt = "You are a travel intent parser for TripAI (Nepal travel recommender). Convert the user's natural language input into a structured JSON query."
inputs = tokenizer(
[
    f"<|im_start|>system\n{system_prompt}<|im_end|>\n<|im_start|>user\nPokhara for 3 days but I'm stressed, something calm, no trekking<|im_end|>\n<|im_start|>assistant\n"
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 512, use_cache = True)
print(tokenizer.decode(outputs[0], skip_special_tokens = True))

### 7. Save Adapter & Export to GGUF
We can save the model adapter or merge and export directly to GGUF format for Ollama deployment.

In [ ]:
# Option A: Save LoRA Adapter only
model.save_pretrained_lora("tripai_nlp_adapter")

# Option B: Merge and Export directly to GGUF (4-bit quantization)
model.save_pretrained_gguf("tripai_qwen_q4", tokenizer, quantization_method = "q4_k_m")

### 8. Deploying on Ollama Locally

Once you have downloaded the exported GGUF file (`tripai_qwen_q4-unsloth.gguf`) from Colab, deploy it locally:

1. Create a `Modelfile` containing:
   ```dockerfile
   FROM ./tripai_qwen_q4-unsloth.gguf
   # Set parameters
   PARAMETER temperature 0.1
   ```
2. Run the command in terminal:
   ```bash
   ollama create tripai-nlp -f Modelfile
   ```
3. Run and test the new model:
   ```bash
   ollama run tripai-nlp
   ```